# RoPE: Math Derivation, Relative Position Proof & CuTeDSL

## 1. Where RoPE Sits in Inference

Diagram showing RoPE's position: after Q/K projection and QK-norm, before attention:

```
x ∈ ℝ^{B×T×4096}  (from RMSNorm)
  │
  ├─ Q = x @ W_Q.T → q_norm → RoPE(Q, positions)  →  Q_rope ∈ ℝ^{B×T×32×128}
  ├─ K = x @ W_K.T → k_norm → RoPE(K, positions)  →  K_rope ∈ ℝ^{B×T×8×128}
  └─ V = x @ W_V.T                                  →  V      ∈ ℝ^{B×T×8×128}
                                                           ↓
                                     attention(Q_rope, K_rope, V)  →  output
```

Note: V is NOT rotated. Only Q and K get RoPE.

## 2. I/O Specification

| Field | Value |
|---|---|
| Input Q | q ∈ ℝ^{B × T × H_q × D_h} where H_q=32, D_h=128 |
| Input K | k ∈ ℝ^{B × T × H_k × D_h} where H_k=8, D_h=128 |
| Input dtype | bfloat16 |
| What produced inputs | Q/K projection GEMMs + QK-norm RMSNorm |
| Position indices | m ∈ {0, 1, ..., T-1} (integer token positions in sequence) |
| Frequency table | cos_table, sin_table ∈ ℝ^{T_max × (D_rope/2)} precomputed at model load |
| Output Q_rope | same shape as Q — first D_rope=64 dims rotated, last 64 unchanged |
| Output K_rope | same shape as K — same split |
| What consumes outputs | attention kernel: Q_rope @ K_rope.T / sqrt(D_h) |
| Computation type | element-wise 2D rotation per dimension pair — no cross-pair communication |

## 3. The Formula (2D Case First)

Start with 2D to build intuition. A 2D vector [x₀, x₁] at position m is rotated by angle m·θ:

$$\begin{bmatrix} x_0' \\ x_1' \end{bmatrix} = R(m\theta) \begin{bmatrix} x_0 \\ x_1 \end{bmatrix} = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}$$

Expanding:
$$x_0' = x_0 \cos(m\theta) - x_1 \sin(m\theta)$$
$$x_1' = x_0 \sin(m\theta) + x_1 \cos(m\theta)$$

**General D-dimensional case:** Split D into D/2 independent pairs. Pair k (dims 2k and 2k+1):
$$\theta_k = \frac{1}{\text{base}^{2k/D_{rope}}}$$

$$\begin{bmatrix} x_{2k}' \\ x_{2k+1}' \end{bmatrix} = \begin{bmatrix} \cos(m\theta_k) & -\sin(m\theta_k) \\ \sin(m\theta_k) & \cos(m\theta_k) \end{bmatrix} \begin{bmatrix} x_{2k} \\ x_{2k+1} \end{bmatrix}$$

Note: Qwen3 uses D_rope=64 (only 32 pairs per head), base=1,000,000. Dims 64–127 are NoPE (no rotation).

## 4. Full Derivation: Why RoPE Encodes Relative Positions

This is the key mathematical result. Start from scratch.

**Claim:** The dot product of a rotated query at position m and a rotated key at position n depends only on their relative distance (n−m), not on their absolute positions.

**Setup:** Let R(mθ) denote the block-diagonal rotation matrix for position m. For d=2:
$$R(m\theta) = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix}$$

This is a standard 2D rotation. Rotation matrices have the composition property:
$$R(\alpha)^T R(\beta) = R(\beta - \alpha)$$
Proof: $R(\alpha)^T = R(-\alpha)$ (inverse rotation), and $R(-\alpha) R(\beta) = R(\beta - \alpha)$.

**The dot product:**
$$Q_{rope}[m]^T K_{rope}[n] = (R(m)Q)^T (R(n)K)$$
$$= Q^T R(m)^T R(n) K$$
$$= Q^T R(n-m) K$$

The result depends on Q, K, and the relative offset (n−m) — not on m or n individually.

**Consequence:** Tokens at positions (3, 7) have the same rotation offset as tokens at (103, 107): both see Δ=4. A model trained with sequences of length T can extrapolate to longer sequences because the relative offsets it learned (Δ=1 means adjacent, Δ=100 means far away) remain valid regardless of absolute position.

**Why this beats learned positional embeddings:**

| Property | Learned PE | RoPE |
|---|---|---|
| Attention score depends on | Absolute positions m, n | Relative offset n−m only |
| Extrapolates to longer context | ❌ | ✅ |
| Extra parameters | T_max × D | 0 |
| Context extension method | Retrain | Increase base or scaling |

## 5. Symbol Table

| Symbol | Definition | Python / PyTorch | CuTeDSL | CUDA / Triton |
|---|---|---|---|---|
| q, k | Query/key tensors after projection | `q: Tensor[B,T,H,D_h]` | `mQ, mK: cute.Tensor` | `__nv_bfloat16* q` |
| m | Token position (absolute) | `positions: Tensor[T]` | `seq_pos = cute.block_idx('x')` | `int seq_pos = blockIdx.x` |
| D_rope | Number of dims to rotate (64 in Qwen3) | `rope_head_dim = 64` | `D_ROPE = 64` | `const int D_ROPE = 64` |
| D_h | Full head dim (128 in Qwen3) | `head_dim = 128` | `D = 128` | `const int D = 128` |
| θ_k | Frequency for pair k | `freqs[k] = 1/base**(2k/D_rope)` | precomputed table | precomputed table |
| cos_m, sin_m | Rotation coefficients for position m, pair k | `cos_table[m, k], sin_table[m, k]` | `mFreqCos[seq_pos, k], mFreqSin[...]` | loaded from freq table |
| R(m) | Rotation matrix for position m | N/A (implicit via view_as_complex) | implicit in pair-wise ops | applied pair by pair |
| x₀, x₁ | Even/odd elements of a pair | `x[..., 2k], x[..., 2k+1]` | `mX[row, head, 2*k], mX[row, head, 2*k+1]` | `x_even, x_odd` |
| NoPE | Dims not rotated (64–127 in Qwen3) | `x[..., D_rope:]` (copied unchanged) | same — no rotation applied | `if k < D_ROPE/2:` |

In [ ]:
import torch, math

torch.manual_seed(0)
B, T, H, D_h = 1, 4, 32, 128
D_rope = 64   # Qwen3: only first 64 dims rotated
base   = 1_000_000   # Qwen3 rope_theta

# Step 0: input (one head's Q tensor for simplicity)
q = torch.randn(B, T, H, D_h)

# ── Level 1: what RoPE does ─────────────────────────────────────
# q_rope = rope(q, positions, D_rope, base)

# ── Level 2: split into RoPE region and NoPE region ─────────────
# q_rope_region = rotate(q[..., :D_rope], positions, base)
# q_nope_region = q[..., D_rope:]  # unchanged
# q_rope = concat([q_rope_region, q_nope_region], dim=-1)

# ── Level 3: how to rotate D_rope dims ──────────────────────────
# split into D_rope/2 pairs; rotate each pair by (m * theta_k)

# ── Level 4: precompute frequency table ─────────────────────────
num_pairs = D_rope // 2   # = 32 pairs
k_indices = torch.arange(num_pairs, dtype=torch.float32)
thetas    = 1.0 / (base ** (2 * k_indices / D_rope))   # shape [32]
print(f"theta_0 (fastest): {thetas[0]:.6f}  (full rotation every ~{2*math.pi/thetas[0]:.0f} tokens)")
print(f"theta_31 (slowest): {thetas[-1]:.8f}  (full rotation every ~{2*math.pi/thetas[-1]:.0f} tokens)")

# ── Level 5: for each position, compute cos and sin ─────────────
positions  = torch.arange(T, dtype=torch.float32)   # [T]
angles     = torch.outer(positions, thetas)           # [T, 32]  m * theta_k for all m,k
cos_table  = torch.cos(angles)   # [T, 32]
sin_table  = torch.sin(angles)   # [T, 32]

# ── Level 6: apply rotation to each pair ────────────────────────
q_rope = q.clone()
for ki in range(num_pairs):
    i0, i1 = 2*ki, 2*ki+1
    c = cos_table[:, ki].view(1, T, 1, 1)   # broadcast to [B,T,H,1]
    s = sin_table[:, ki].view(1, T, 1, 1)
    x0 = q[..., i0:i0+1]
    x1 = q[..., i1:i1+1]
    q_rope[..., i0:i0+1] = x0 * c - x1 * s
    q_rope[..., i1:i1+1] = x0 * s + x1 * c
# dims D_rope..D_h are already copied (q.clone())

# ── Verify: norm is preserved (rotation doesn't change length) ──
norm_before = q[..., :D_rope].norm(dim=-1)
norm_after  = q_rope[..., :D_rope].norm(dim=-1)
err = (norm_before - norm_after).abs().max()
print(f"\nNorm preserved (max diff): {err:.2e}  (should be ~0)")
assert err < 1e-4, "rotation changed vector length — bug!"
print("✓ Rotation is an isometry: norm unchanged")

In [ ]:
# Verify: dot(RoPE(q, pos=m), RoPE(k, pos=n)) depends only on (n-m)
def rope_single(x, m, thetas):
    """Apply RoPE to a single vector x at position m."""
    out = x.clone()
    for ki, theta in enumerate(thetas):
        i0, i1 = 2*ki, 2*ki+1
        angle = m * theta
        c, s = math.cos(angle), math.sin(angle)
        x0, x1 = x[i0].item(), x[i1].item()
        out[i0] = x0*c - x1*s
        out[i1] = x0*s + x1*c
    return out

thetas_list = thetas.tolist()
q_vec = torch.randn(D_rope)
k_vec = torch.randn(D_rope)

# Dot products at (m=0, n=5) and (m=100, n=105) — same relative offset Δ=5
def dot(a, b): return (a * b).sum().item()

dp_05   = dot(rope_single(q_vec, 0,   thetas_list), rope_single(k_vec, 5,   thetas_list))
dp_100  = dot(rope_single(q_vec, 100, thetas_list), rope_single(k_vec, 105, thetas_list))
dp_1000 = dot(rope_single(q_vec, 1000,thetas_list), rope_single(k_vec, 1005,thetas_list))

print("Relative position property: dot products with Δ=5 at different absolute positions")
print(f"  (pos=0,   pos=5):    {dp_05:.6f}")
print(f"  (pos=100, pos=105):  {dp_100:.6f}")
print(f"  (pos=1000,pos=1005): {dp_1000:.6f}")
print(f"  Max diff: {max(abs(dp_05-dp_100), abs(dp_05-dp_1000)):.2e}")
print("✓ All equal (up to float precision) — relative position property holds")

## 6. Frequency Clock Table for Qwen3-8B

For Qwen3: D_rope=64, base=1,000,000, so 32 pairs per head. Show:

| Pair k | θ_k | Period (tokens) | What it encodes |
|---|---|---|---|
| 0 | 1.000 | ~6 | Immediately adjacent |
| 4 | 0.032 | ~196 | Short phrases |
| 8 | 0.001 | ~6,283 | Paragraphs |
| 16 | 1e-6 | ~6.28M | Very long context |
| 31 | ~0 | → ∞ | Long-context anchor |

With base=10,000 (standard): pair 31 period ≈ 62,800 tokens → wraps within 64K context.
With base=1,000,000 (Qwen3): pair 31 period ≈ 6.28M tokens → no wrap within 128K context.

## 7. Data Flow at Batch Scale

At decode (B=32 users, T=1 new token per user):
- Q shape after projection: [32, 1, 32, 128] → 32×1×32 = 1,024 head-position pairs to rotate
- Threads per pair: 64 (one per dimension in rope region) or 32 if unrolling 2 elements/thread
- Total threads needed: 1,024 × 64 = 65,536 → 65,536/64 = 1,024 blocks
- RTX 4080: 76 SMs × 16 blocks per SM = 1,216 blocks simultaneously → RoPE at decode finishes in ~1 wave

At prefill (B=1 user, T=2048 tokens):
- Q shape: [1, 2048, 32, 128] → 65,536 head-position pairs
- Total blocks: 65,536 → takes many waves but no synchronization needed

RoPE is never the bottleneck. The kernel launch overhead (5–20µs) often exceeds the compute time at decode. Fix: fuse into Q/K projection GEMM epilogue.

In [ ]:
# CuTeDSL kernel: one block per (sequence_position × head)
# Thread k handles dimension pair k (dims 2k and 2k+1)

# @cute.kernel
# def rope_sm89(mQ, mFreqCos, mFreqSin, mOut):
#     seq_pos = cute.block_idx('x')    # which token position
#     head    = cute.block_idx('y')    # which attention head
#     k       = cute.thread_idx('x')  # which dimension pair (0..31)
#
#     # Load the two adjacent dimensions forming pair k
#     # mQ shape: [T, H, D_h] where D_h=128
#     x0 = mQ[seq_pos, head, 2*k]       # even dim — BF16 load
#     x1 = mQ[seq_pos, head, 2*k + 1]   # odd  dim — BF16 load
#
#     # Load precomputed rotation angles (computed once at model load)
#     # mFreqCos/mFreqSin shape: [T_max, 32]
#     cos_m = mFreqCos[seq_pos, k]
#     sin_m = mFreqSin[seq_pos, k]
#
#     # Apply 2D rotation:
#     # [x0']   [cos  -sin] [x0]
#     # [x1'] = [sin   cos] [x1]
#     mOut[seq_pos, head, 2*k]     = x0 * cos_m - x1 * sin_m   # 3 FLOPs
#     mOut[seq_pos, head, 2*k + 1] = x0 * sin_m + x1 * cos_m   # 3 FLOPs
#     # Total: 6 FLOPs per pair, zero cross-thread communication

print("CuTeDSL RoPE kernel properties:")
print("  Threads are 100% independent — no reduction, no SMEM, no barriers")
print("  Each thread: 2 loads (x0, x1) + 2 loads (cos, sin) + 2 writes = 6 FLOPs")
print("  Arithmetic intensity: 6 FLOPs / (6 × 2 bytes) = 0.5 FLOP/byte")
print("  Always memory-bound regardless of hardware")
print()
print("NoPE handling (dims 64..127):")
print("  Launch only D_rope/2 = 32 threads per head")
print("  Dims 64..127 are written unchanged in a separate pass or fused copy")
print("  Or: fuse RoPE into GEMM epilogue — write all 128 dims once from registers")

## 8. Production RoPE Kernels

| Library | API | Key feature | Source |
|---|---|---|---|
| HuggingFace | `apply_rotary_pos_emb()` | PyTorch, view_as_complex trick | `modeling_qwen3.py` |
| FlashInfer | `apply_rope_inplace(q, k, indptr, offsets, rope_theta=1e6)` | In-place, paged KV, variable-length | `flashinfer-ai/flashinfer rope.h` |
| FlashInfer | `apply_rope_with_cos_sin_cache(...)` | Precomputed cos/sin table, avoids recomputing trig | same |
| vLLM | `apply_rotary_emb()` Triton or CUDA kernel | Dispatches to Triton for portability | `vllm/model_executor/layers/rotary_embedding.py` |
| SGLang | FlashInfer's `apply_rope_inplace` directly | No custom kernel | `srt/models/qwen3.py` |
| TRT-LLM | Fused inside FusedMHA plugin | Not a separate kernel in TRT plan | `tensorrt_llm/plugin/` |

**Kernel fusion with QKV projection:**
Q and K are the output of a GEMM (q_proj, k_proj). Fusing RoPE into the GEMM epilogue:
- GEMM computes Q in registers (rC)
- Apply RoPE in registers: rotate pairs in-place
- Write rotated Q to HBM once
- Saves: one HBM write + one HBM read of Q (at decode: [1,32,128] = 8 KB, tiny but eliminates a kernel launch)

Most production systems apply this fusion at decode via FlashInfer's fused attention primitives.

## One-Sentence Takeaway

RoPE is mathematically elegant (the relative position property Q^T R(m)^T R(n) K = Q^T R(n−m) K is proven by the rotation composition law), computationally trivial (6 FLOPs per pair, zero synchronization, always memory-bound at 0.5 F/B), and the main engineering concern is not the rotation arithmetic itself but eliminating the kernel launch overhead by fusing it into the Q/K projection epilogue — which is why production systems like FlashInfer expose `apply_rope_inplace` while TRT-LLM never surfaces RoPE as a standalone operation at all.